In [1]:
import torch
from torch.utils.data import Dataset

PAD_ID = 2

def flatten_state(cur_seq):
    flat = []
    for seq in cur_seq:
        for b in seq:
            flat.append(1 if b > 0 else 0)
    return torch.tensor(flat, dtype=torch.long)

class DPODataset(Dataset):
    def __init__(self, samples):
        self.data = []
        for seq, chosen, rejected in samples:
            x = flatten_state(seq)
            self.data.append((x, chosen, rejected))

    def __len__(self): return len(self.data)
    def __getitem__(self, i): return self.data[i]

In [2]:
import torch.nn as nn
import torch.nn.functional as F

class PolicyOnlyTransformer(nn.Module):
    def __init__(
        self,
        vocab_size=3,         # tokens {0,1,PAD_ID}
        pad_id=PAD_ID,
        max_len=4096,
        d_model=256,
        nhead=8,
        nlayers=4,
        dim_ff=512,
        dropout=0.1,
    ):
        super().__init__()
        self.pad_id = pad_id
        self.token_emb = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.pos_emb   = nn.Embedding(max_len, d_model)  # learned positional embeddings

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_ff,
            dropout=dropout, activation="gelu", batch_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=nlayers)
        self.norm    = nn.LayerNorm(d_model)
        self.head    = nn.Linear(d_model, 2)  # logits for {0,1}

    def forward(self, x, pad_mask):
        """
        x:        [B, T] int64 in {0,1,pad_id}
        pad_mask: [B, T] bool, True where PAD (ignored by attention)
        """
        B, T = x.size()
        pos = torch.arange(T, device=x.device).unsqueeze(0).expand(B, T)  # [B, T]
        h = self.token_emb(x) + self.pos_emb(pos)                         # [B, T, D]

        # Mask pads inside the encoder
        h = self.encoder(h, src_key_padding_mask=pad_mask)                # [B, T, D]

        # Pool from the last non-pad token per sequence
        valid = ~pad_mask                                # True where real token
        lengths = valid.long().sum(dim=1) - 1            # index of last real token
        lengths = lengths.clamp(min=0)                   # guard against all-pad (shouldn't happen)
        idx = lengths.view(B, 1, 1).expand(B, 1, h.size(-1))
        last_h = h.gather(1, idx).squeeze(1)             # [B, D]

        logits = self.head(self.norm(last_h))            # [B, 2]
        return logits

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

PAD_ID = 2

class ActorCriticFromPolicyOnly(nn.Module):
    def __init__(self, base: "PolicyOnlyTransformer"):
        super().__init__()
        # reuse everything from your pretrained model
        self.pad_id = base.pad_id
        self.token_emb = base.token_emb
        self.pos_emb   = base.pos_emb
        self.encoder   = base.encoder
        self.norm      = base.norm
        self.pi_head   = base.head          # policy logits [B,2]

        d_model = self.pi_head.in_features
        self.v_head = nn.Linear(d_model, 1) # new critic head

    def forward_features(self, x, pad_mask):
        B, T = x.size()
        pos = torch.arange(T, device=x.device).unsqueeze(0).expand(B, T)
        h = self.token_emb(x) + self.pos_emb(pos)
        h = self.encoder(h, src_key_padding_mask=pad_mask)

        valid = ~pad_mask
        lengths = (valid.long().sum(dim=1) - 1).clamp(min=0)
        idx = lengths.view(B, 1, 1).expand(B, 1, h.size(-1))
        last_h = h.gather(1, idx).squeeze(1)  # [B,D]
        z = self.norm(last_h)
        return z

    def forward(self, x, pad_mask):
        z = self.forward_features(x, pad_mask)
        logits = self.pi_head(z)              # [B,2]
        value  = self.v_head(z).squeeze(-1)   # [B]
        return logits, value


In [4]:
base = PolicyOnlyTransformer()
base.load_state_dict(torch.load("model_long_9.pt", weights_only=True))
ac = ActorCriticFromPolicyOnly(base)

In [5]:
import torch as th
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor

class PretrainedTransformerExtractor(BaseFeaturesExtractor):
    def __init__(self, observation_space, pretrained_state_dict_path, d_model=256, **arch_kwargs):
        super().__init__(observation_space, features_dim=d_model)

        max_t = observation_space["tokens"].shape[0]

        # create a PolicyOnlyTransformer with same hyperparams as pretrained
        self.base = PolicyOnlyTransformer(
            vocab_size=3, pad_id=PAD_ID, max_len=max_t,
            d_model=d_model,
            nhead=arch_kwargs.get("nhead", 8),
            nlayers=arch_kwargs.get("nlayers", 4),
            dim_ff=arch_kwargs.get("dim_ff", 512),
            dropout=arch_kwargs.get("dropout", 0.0),  # often better for PPO fine-tune
        )
        self.base.load_state_dict(torch.load(pretrained_state_dict_path, weights_only=True), strict=True)

        # IMPORTANT: remove policy head, return features only
        # We'll use everything except head
        self.base.head = nn.Identity()

    def forward(self, obs):
        x = obs["tokens"].long()
        pad_mask = obs["mask"].bool()

        # replicate your forward until last_h
        B, T = x.size()
        pos = th.arange(T, device=x.device).unsqueeze(0).expand(B, T)
        h = self.base.token_emb(x) + self.base.pos_emb(pos)
        h = self.base.encoder(h, src_key_padding_mask=pad_mask)

        valid = ~pad_mask
        lengths = (valid.long().sum(dim=1) - 1).clamp(min=0)
        idx = lengths.view(B, 1, 1).expand(B, 1, h.size(-1))
        last_h = h.gather(1, idx).squeeze(1)

        return self.base.norm(last_h)  # [B, d_model]

In [8]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import copy
from cnf_automator import generate_encoding, verify_assignment

PAD_ID = 2

def flatten_state_bits(cur_seq):
    flat = []
    for seq in cur_seq:
        for b in seq:
            flat.append(1 if b > 0 else 0)
    return flat

def add(seq, next_bit):
    first_len = len(seq[0])
    added = False
    for i in range(4):
        if (len(seq[i]) < first_len):
            new_element = (next_bit, )
            new_seq = seq[i] + new_element
            seq[i] = new_seq
            added = True
            break
    if not added:
        new_element = (next_bit, )
        new_seq = seq[0] + new_element
        seq[0] = new_seq
    return seq

class TurynSatEnv(gym.Env):
    metadata = {"render_modes": []}

    def __init__(
        self,
        r,
        max_t,
        step_reward=0.01,
        unsat_penalty=-1.0,
    ):
        super().__init__()
        self.r = r
        self.max_t = max_t
        self.step_reward = step_reward
        self.unsat_penalty = unsat_penalty

        # Action: choose next bit
        self.action_space = spaces.Discrete(2)

        # Observation: padded tokens + pad mask
        self.observation_space = spaces.Dict({
            "tokens": spaces.Box(
                low=0, high=PAD_ID, shape=(max_t,), dtype=np.int64
            ),
            "mask": spaces.Box(
                low=0, high=1, shape=(max_t,), dtype=np.int8
            ),
        })

        # Build CNF once
        self.cnf, self.varmap, self.pmap_lags, self.pool = generate_encoding(r)

        self.seq = None
        self.done = False

    def _make_obs(self):
        bits = flatten_state_bits(self.seq)
        t = len(bits)

        tokens = np.full((self.max_t,), PAD_ID, dtype=np.int64)
        mask = np.ones((self.max_t,), dtype=np.int8)

        if t > 0:
            t = min(t, self.max_t)
            tokens[:t] = np.array(bits[:t], dtype=np.int64)
            mask[:t] = 0

        return {"tokens": tokens, "mask": mask}

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.seq = [(1,), (1,), (1,), (1,)]
        sat, core = verify_assignment(self.seq, self.varmap, self.cnf)
        self.done = bool(core)
        return self._make_obs(), {}

    def step(self, action):
        if self.done:
            raise RuntimeError("step() called after episode done")

        # apply action
        self.seq = add(self.seq, int(action))

        sat, core = verify_assignment(self.seq, self.varmap, self.cnf)
        if core:
            self.done = True
            reward = self.unsat_penalty
            terminated = True
        else:
            reward = self.step_reward
            terminated = False

        truncated = False
        info = {"sat": int(not bool(core))}
        return self._make_obs(), reward, terminated, truncated, info

In [9]:
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env

r = 13
max_t = 4096  # set to your max flattened length upper bound

env = make_vec_env(lambda: TurynSatEnv(r=r, max_t=max_t), n_envs=4)

policy_kwargs = dict(
    features_extractor_class=PretrainedTransformerExtractor,
    features_extractor_kwargs=dict(
        pretrained_state_dict_path="model_long_9.pt",
        d_model=256,
        nhead=8, nlayers=4, dim_ff=512, dropout=0.0
    ),
    # these are SB3's heads on top of transformer features
    net_arch=dict(pi=[128], vf=[128]),
)

model = PPO(
    "MultiInputPolicy",
    env,
    policy_kwargs=policy_kwargs,
    learning_rate=1e-5,   # start small for fine-tune
    n_steps=2048,
    batch_size=256,
    n_epochs=4,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.01,
    vf_coef=0.5,
    max_grad_norm=1.0,
    verbose=1,
)

model.learn(total_timesteps=50_000)
model.save("model_long_9_ppo.pt")

Using cpu device
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 22.8     |
|    ep_rew_mean     | -0.782   |
| time/              |          |
|    fps             | 17       |
|    iterations      | 1        |
|    time_elapsed    | 461      |
|    total_timesteps | 8192     |
---------------------------------


: 

: 

In [7]:
%pip install stable-baselines3

  Obtaining dependency information for stable-baselines3 from https://files.pythonhosted.org/packages/df/cc/a3038d3833f329dcd03b2dce8b778e4b41044caff88b48429473b8629623/stable_baselines3-2.7.1-py3-none-any.whl.metadata
  Obtaining dependency information for torch<3.0,>=2.3 from https://files.pythonhosted.org/packages/61/d8/15b9d9d3a6b0c01b883787bd056acbe5cc321090d4b216d3ea89a8fcfdf3/torch-2.10.0-cp311-none-macosx_11_0_arm64.whl.metadata
  Obtaining dependency information for sympy>=1.13.3 from https://files.pythonhosted.org/packages/a2/09/77d55d46fd61b4a135c444fc97158ef34a095e5681d0a6c10b75bf356191/sympy-1.14.0-py3-none-any.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.0/188.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.4/79.4 MB 5.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 6.0 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: sympy
    Found existing installation: sympy 1.11.1
    Uninstalling sympy-1.